# Stage1 FINRA 场外卖空标记占比差

唯一新增信息 q=SOXL场外ShortVolume/TotalVolume−SOXX同口径占比。B/C共享availability、报告制度和总量背景，B的q输入固定为0。原非线性shared-head Stage1、标签及完整控制链冻结。不能解释为shortinterest、情绪或operational shorting复现。

[冻结计划](plan.json) / [中文结论](conclusion.md)。当前骨架禁止拟合。

In [1]:
import os, sys, json
from pathlib import Path
root = Path.cwd()
while not (root / "AGENTS.md").exists():
    if root.parent == root:
        raise RuntimeError("workspace not found")
    root = root.parent
sys.path.insert(0, str(root))
os.environ["MLFLOW_ALLOW_FILE_STORE"] = "true"
import qlib
from qlib.data import D
from qlib.workflow import R
from src.workflow.finra_allocation import prepare, run
card = root / "research/stage1_finra_short_mix_allocation_20261018"
plan = json.loads((card / "plan.json").read_text())
# prepare invokes qlib.init, DatasetH.prepare and verified Recorder inputs.
plan["status"]

Environment notice: installed Gym emits its existing unmaintained-version warning.


'frozen_pending_model_design_review'

## 来源与无拟合预检

原来源serialized Dataset缺少handler缓存，保持原archive封存。显式读取hash验证的float64 Recorder panel，仅原6个source字段，经DataHandlerLP.from_df重建DatasetH并prepare，再公式与派生Qlib float32读回。新卡Dataset保存前启用recursive dump_all，保存可prepare缓存，不声称StaticDataLoader可setup重建。严格保留原主样本和末端NaT不可执行行，不因来源缺失过滤。必须等来源独评与计划冻结后才能执行真实预检。

In [2]:
prepared = prepare(root=root, card=card) if plan.get("frozen_before_fits") else None
prepared["receipt"] if prepared is not None else {"status": "draft_pending_source_result", "fits": 0, "accounts": 0}

{'plan_sha256': '017535b8a9708c80475e04cfaf869a4cd70a42ff369d8b8fe1c4062ef4d7cc0e',
 'primary_rows': 3014,
 'auxiliary_rows': 20741,
 'primary_last_mature': '2022-12-30 09:31:00-05:00',
 'aux_first_signal': '2019-11-25 00:00:00',
 'aux_last_signal': '2022-12-22 00:00:00',
 'aux_observations_end_max': '2022-12-30 00:00:00',
 'source_hashes_checked': True,
 'source_api': ['Recorder.load_object', 'DatasetH.prepare'],
 'main_inputs': 39,
 'additional_allocation_inputs': 6,
 'common_inputs': 16,
 'account_sessions': 944,
 'early_fields_exact_source_equality': ['early_ret_soxl',
  'early_ret_soxx',
  'early_range_soxl',
  'early_range_soxx',
  'early_from_low_soxl',
  'early_from_low_soxx',
  'early_volume_soxl',
  'early_volume_soxx',
  'early_bar_count_soxl',
  'early_bar_count_soxx'],
 'reused_reference_accounts': 15,
 'active_bundle_and_classifier_hashes_verified': True,
 'initialization': [{'seed': 71,
   'original_tensors_identical': True,
   'paired_initial_tensors_identical': True,
 

## 固定训练与完整经济比较

6个固定seed×arm拟合；全部0/2/5成本账户，无信息分数筛选。6新账户+15严格复用账户=21。5个required comparator全部财富及配对CI门，SOXX年度/rolling70%门，close/采样点DD与CVaR门；重叠历史不独立。

In [3]:
if os.environ.get("FINRA_ALLOCATION_EXECUTE") == "1":
    result = run(root=root, card=card, prepared=prepared)
else:
    result = json.loads((card / "summary.json").read_text())
result

completed fit B_FINRAcontext_71
completed fit B_FINRAcontext_172
completed fit B_FINRAcontext_273
completed fit C_FINRA_short_mix_71
completed fit C_FINRA_short_mix_172
completed fit C_FINRA_short_mix_273
completed native account B_FINRAcontext_0
completed native account B_FINRAcontext_2
completed native account B_FINRAcontext_5
completed native account C_FINRA_short_mix_0
completed native account C_FINRA_short_mix_2
completed native account C_FINRA_short_mix_5


Qlib emitted RuntimeWarning: Mean of empty slice 5676 times; repeated identical library warnings were compacted. Native account audit evidence is retained in the Recorder.


{'experiment': 'stage1_finra_short_mix_allocation_20261018',
 'plan_sha256': '017535b8a9708c80475e04cfaf869a4cd70a42ff369d8b8fe1c4062ef4d7cc0e',
 'decision': 'not_promoted_fixed_FINRA_short_mix_v1',
 'objective_achieved': False,
 'gates': {'net_relative_wealth_all_costs': False,
  'sampled_drawdown_30pct': False,
  'cvar_no_worse_soxx': True,
  'minimum_complete_years': True,
  'complete_annual_70pct': False,
  'rolling126_70pct': False,
  'paired_block_lower_positive': False,
  'fresh_fullminute_risk_qualified': False,
  'qualified_independent_forward': False},
 'coarse_economic_gates_pass': False,
 'new_fits': 6,
 'new_native_accounts': 6,
 'reused_native_accounts': 15,
 'total_comparison_accounts': 21,
 'sessions_each': 944,
 'recorder_id': '8e87af53d57a4022bcf19b5dbc690e33',
 'experiment_id': '406826129487703844',
 'recorder_uri': '/Users/rziiiii/dev/workspace/research/stage1_finra_short_mix_allocation_20261018/qlib_recorder',
 'recorder_local_dir': '/Users/rziiiii/dev/workspace/re

## 一次固定失败归因

以下cell在拟合/账户完成后以独立短分析notebook执行（session96244，exit0），仅复用本轮Recorder。原训练、账户和bootstrap均未重跑；全部年/季度、提议、状态固定汇总，无新增HTTP或窗口搜索。


In [1]:
import os, json, hashlib
from pathlib import Path
import numpy as np
import pandas as pd
import qlib
from src.core.recorder_sources import load_verified_recorder_objects
root = Path('/Users/rziiiii/dev/workspace')
card = root / 'research/stage1_finra_short_mix_allocation_20261018'
summary = json.loads((card / 'summary.json').read_text())
manifest = json.loads((card / 'verification_manifest.json').read_text())
qlib.init(provider_uri=summary['provider_uri'], region='us', kernels=1)
keys = ['analysis/ensembles', 'analysis/smoothed_targets', 'analysis/exposure_summary',
        'analysis/annual_metrics', 'analysis/quarterly_metrics', 'analysis/metrics', 'analysis/uncertainty']
keys += [f'native/{family}_{cost}' for family in ['B_FINRAcontext', 'C_FINRA_short_mix'] for cost in [0,2,5]]
hashes = {key: next(x['sha256'] for x in manifest['entries'] if x['path'].endswith('/artifacts/'+key)) for key in keys}
objects, audit = load_verified_recorder_objects(card, expected_recorder_id=summary['recorder_id'], artifacts=keys,
    expected_manifest_sha256=hashlib.sha256((card/'verification_manifest.json').read_bytes()).hexdigest(),
    expected_artifact_sha256=hashes)
# Complete predeclared years and quarters; no winner-selected subwindows.
periods = pd.concat([objects['analysis/annual_metrics'], objects['analysis/quarterly_metrics']], ignore_index=True)
comparisons = []
for comparator in ['B_FINRAcontext', 'B_shared', 'SOXX']:
    fields = ['kind','period','slippage_bps','complete_period','sessions','return_pct','terminal_wealth']
    left = periods.loc[periods.family.eq('C_FINRA_short_mix'), fields]
    right = periods.loc[periods.family.eq(comparator), fields]
    joined = left.merge(right, on=['kind','period','slippage_bps'], suffixes=('_candidate','_reference'), validate='one_to_one')
    assert joined.complete_period_candidate.equals(joined.complete_period_reference)
    assert joined.sessions_candidate.equals(joined.sessions_reference)
    for row in joined.itertuples():
        relative = row.terminal_wealth_candidate / row.terminal_wealth_reference
        comparisons.append({'kind':row.kind,'period':row.period,'slippage_bps':row.slippage_bps,
            'complete_period':bool(row.complete_period_candidate),'sessions':row.sessions_candidate,
            'candidate':'C_FINRA_short_mix','comparator':comparator,
            'candidate_return_pct':row.return_pct_candidate,'comparator_return_pct':row.return_pct_reference,
            'relative_wealth':relative,'candidate_wins':bool(relative>1+1e-8)})
period_table = pd.DataFrame(comparisons)
assert len(period_table)==180
period_table.to_csv(card/'failure_period_comparison.csv',index=False)
period_summary = period_table.groupby(['kind','slippage_bps','comparator','complete_period'],as_index=False).agg(periods=('candidate_wins','size'),wins=('candidate_wins','sum'))
period_summary['positive_fraction'] = period_summary.wins/period_summary.periods
period_summary.to_csv(card/'failure_period_summary.csv',index=False)

# Raw and smooth are signal proposals. Controlled/held are different moments,
# so this is a descriptive transmission comparison, not additive cash-drag attribution.
ensembles, smoothed = objects['analysis/ensembles'], objects['analysis/smoothed_targets']
exposure = objects['analysis/exposure_summary']
proposal_rows, native_audits = [], []
books = {}
for family in ['B_FINRAcontext','C_FINRA_short_mix']:
    for cost in [0,2,5]:
        book = objects[f'native/{family}_{cost}']; path=book['authority']; books[(family,cost)]=book
        dates=pd.DatetimeIndex(path.signal_date)
        raw=ensembles[family].reindex(dates); smooth=smoothed[family].reindex(dates); target=book['targets']
        assert len(path)==944 and pd.DatetimeIndex(target.signal_date).equals(dates)
        assert np.isfinite(raw[['soxl_weight','soxx_weight','cash_weight']]).all().all()
        row={'family':family,'slippage_bps':cost,'sessions':944}
        for stage, frame in [('raw',raw),('smoothed',smooth),('controlled_or_hold',target)]:
            row[stage+'_mean_soxl_weight']=float(frame.soxl_weight.mean())
            row[stage+'_mean_cash_weight']=float(frame.cash_weight.mean())
            row[stage+'_mean_nominal_beta']=float((3*frame.soxl_weight+frame.soxx_weight).mean())
        row.update(actual_mean_soxl_weight=float(path.ending_soxl_weight.mean()),
            actual_mean_cash_weight=float(path.ending_cash_weight.mean()),
            actual_mean_nominal_beta=float((3*path.ending_soxl_weight+path.ending_soxx_weight).mean()),
            nonzero_risk_state_days=int(path.risk_state.ne(0).sum()),
            open_execute_days=int(path.execute_open_target.sum()),recovery_days=int(path.recovery_action.sum()),
            mean_beta_cap=float(path.current_beta_cap.mean()))
        saved=exposure.loc[exposure.family.eq(family)&exposure.slippage_bps.eq(cost)].iloc[0]
        assert np.isclose(row['actual_mean_nominal_beta'],saved.mean_ending_nominal_beta,atol=1e-12)
        proposal_rows.append(row)
        native_audits.append({'family':family,'cost':cost,'status':book['audit']['status'],
            'points':len(book['points']),'max_absolute_relative_nav_error':float(book['points'].relative_error.abs().max())})
proposal_table=pd.DataFrame(proposal_rows)
proposal_table.to_csv(card/'failure_proposal_exposure.csv',index=False)
exposure.loc[exposure.family.isin(['B_FINRAcontext','C_FINRA_short_mix','B_shared','SOXX'])].to_csv(card/'failure_reference_exposure.csv',index=False)

state_pairs=[]
def state_comparison(left_key,right_key,role):
    left=books[left_key]['authority'];right=books[right_key]['authority']
    pd.testing.assert_frame_equal(left[['signal_date','trade_date','exit_date']],right[['signal_date','trade_date','exit_date']])
    return {'role':role,'left_family':left_key[0],'left_cost':left_key[1],'right_family':right_key[0],'right_cost':right_key[1],
        'sessions':944,'risk_state_different_days':int(left.risk_state.ne(right.risk_state).sum()),
        'risk_clock_different_days':int(left.risk_clock.ne(right.risk_clock).sum()),
        'beta_cap_different_days':int((left.current_beta_cap-right.current_beta_cap).abs().gt(1e-12).sum()),
        'open_execute_different_days':int(left.execute_open_target.ne(right.execute_open_target).sum()),
        'recovery_different_days':int(left.recovery_action.ne(right.recovery_action).sum()),
        'mean_absolute_actual_cash_weight_difference':float((left.ending_cash_weight-right.ending_cash_weight).abs().mean())}
for cost in [0,2,5]:state_pairs.append(state_comparison(('B_FINRAcontext',cost),('C_FINRA_short_mix',cost),'same_cost_B_vs_C'))
for left,right in [(0,2),(0,5),(2,5)]:state_pairs.append(state_comparison(('C_FINRA_short_mix',left),('C_FINRA_short_mix',right),'C_across_costs'))
state_table=pd.DataFrame(state_pairs);state_table.to_csv(card/'failure_state_path_comparison.csv',index=False)
metrics=objects['analysis/metrics'];ci=objects['analysis/uncertainty'];full=[]
for cost in [0,2,5]:
    values=metrics.loc[metrics.slippage_bps.eq(cost)].set_index('family')
    wealth=lambda name:1+float(values.loc[name,'return_pct'])/100
    full.append({'slippage_bps':cost,'C_vs_Bcontext_relative_wealth':wealth('C_FINRA_short_mix')/wealth('B_FINRAcontext'),
        'Bcontext_vs_Bshared_relative_wealth':wealth('B_FINRAcontext')/wealth('B_shared'),
        'C_vs_Bshared_relative_wealth':wealth('C_FINRA_short_mix')/wealth('B_shared'),
        'C_vs_SOXX_relative_wealth':wealth('C_FINRA_short_mix')/wealth('SOXX')})
result={'status':'bounded_failure_attribution_completed','source_recorder_id':summary['recorder_id'],
 'source_manifest_sha256':hashlib.sha256((card/'verification_manifest.json').read_bytes()).hexdigest(),
 'inputs':hashes,'new_fit':0,'new_replay':0,'new_source_HTTP':0,
 'all_period_comparisons':len(period_table),'period_summary':period_summary.to_dict('records'),
 'full_window_relations':full,'proposal_transmission':proposal_rows,'state_path_comparisons':state_pairs,
 'native_audits':native_audits,'paired_CI_positive_count':int(ci.lower_bound_positive.sum()),'paired_CI_total':len(ci),
 'five_bps_close_and_sampled_DD_pct':metrics.loc[metrics.family.eq('C_FINRA_short_mix')&metrics.slippage_bps.eq(5),['close_max_drawdown_pct','sampled_point_max_drawdown_pct']].iloc[0].to_dict(),
 'supported':['q improves full-window wealth over matched nuisance background B at all three costs, but every required CI fails.',
  'Background B itself greatly underperforms original B_shared; improvement over this weaker control does not establish stable excess.',
  'Proposal and realized allocation differ and cost-specific states diverge; this is an endogenous path description, not isolated controller causality.',
  'The fixed q_v1 fails total wealth, stability, confidence and 5bp drawdown gates. No sign/window/context/weight rescue is authorized.'],
 'limits':['All market history reused; conditional current-byte FINRA availability is not PIT or independent validation.',
  'Original B_shared raw/smoothed proposals and daily risk states are not copied into this Recorder; only saved aggregate held/controlled exposure and complete period tables are compared here.',
  'Means compare different decision/holding moments and cannot be summed into a precise causal cash-drag decomposition.']}
(card/'failure_attribution.json').write_text(json.dumps(result,ensure_ascii=False,indent=2)+'\n')
{'full_window_relations':full,'proposal_transmission':proposal_rows,'state_path_comparisons':state_pairs,
 'period_summary_2bp':period_summary.loc[period_summary.slippage_bps.eq(2)].to_dict('records'),
 'native_maxerror':max(x['max_absolute_relative_nav_error'] for x in native_audits)}


{'full_window_relations': [{'slippage_bps': 0,
   'C_vs_Bcontext_relative_wealth': 1.3858306629968007,
   'Bcontext_vs_Bshared_relative_wealth': 0.38135504129836134,
   'C_vs_Bshared_relative_wealth': 0.5284935097196805,
   'C_vs_SOXX_relative_wealth': 0.4182999983118389},
  {'slippage_bps': 2,
   'C_vs_Bcontext_relative_wealth': 1.508630476445775,
   'Bcontext_vs_Bshared_relative_wealth': 0.3281677223518207,
   'C_vs_Bshared_relative_wealth': 0.49508382732575196,
   'C_vs_SOXX_relative_wealth': 0.38140598450009516},
  {'slippage_bps': 5,
   'C_vs_Bcontext_relative_wealth': 1.0250005352721956,
   'Bcontext_vs_Bshared_relative_wealth': 0.3947019458848752,
   'C_vs_Bshared_relative_wealth': 0.4045697058049743,
   'C_vs_SOXX_relative_wealth': 0.3080192798548403}],
 'proposal_transmission': [{'family': 'B_FINRAcontext',
   'slippage_bps': 0,
   'sessions': 944,
   'raw_mean_soxl_weight': 0.18601015106321545,
   'raw_mean_cash_weight': 0.4682558864538768,
   'raw_mean_nominal_beta': 0.90376

# Stage1 FINRA 场外卖空标记占比差：固定研究

完成B_FINRAcontext与C_FINRA_short_mix各3个seed，共6次固定300轮拟合。主3014条、辅助20741条；原39主输入、16共享输入、损失、样本与Stage1控制链不变。仅主allocation接6项新坐标，B/C只有q信息不同。

新增B/C各0/2/5bp共6个原生账户；严格复用A、原shared B、SOXX、9097、27baa共15账户；每账户944区间。各账户保留原policy与独立成本状态，原始股数及分红应收经Qlib核对。

|方案|滑点bp|累计净收益%|收盘最大回撤%|CVaR5%|
|---|---:|---:|---:|---:|
|A_active_research_replay|0|229.686|-29.487|-4.389|
|A_active_research_replay|2|280.650|-27.060|-4.211|
|A_active_research_replay|5|41.521|-30.517|-4.026|
|B_shared|0|280.383|-25.709|-4.215|
|B_shared|2|270.223|-25.766|-4.215|
|B_shared|5|265.858|-25.819|-4.204|
|B_FINRAcontext|0|45.061|-28.815|-3.595|
|B_FINRAcontext|2|21.495|-30.411|-3.641|
|B_FINRAcontext|5|44.405|-30.340|-3.682|
|SOXX|0|380.588|-40.863|-4.905|
|SOXX|2|380.568|-40.867|-4.905|
|SOXX|5|380.538|-40.872|-4.906|
|strong9097|0|441.681|-26.812|-4.399|
|strong9097|2|434.269|-26.881|-4.400|
|strong9097|5|423.730|-26.956|-4.401|
|C_FINRA_short_mix|0|101.030|-26.641|-3.754|
|C_FINRA_short_mix|2|83.291|-26.688|-3.668|
|C_FINRA_short_mix|5|48.015|-30.434|-3.563|
|additional27baa|0|485.498|-27.171|-4.454|
|additional27baa|2|477.826|-27.223|-4.455|
|additional27baa|5|466.073|-27.317|-4.457|

决定：**not_promoted_fixed_FINRA_short_mix_v1**。
失败或未取得的门：net_relative_wealth_all_costs, sampled_drawdown_30pct, complete_annual_70pct, rolling126_70pct, paired_block_lower_positive, fresh_fullminute_risk_qualified, qualified_independent_forward。

所有年度、季度、126交易日窗口及每个seed分类诊断均保留；不择窗、择seed或按分类分数取消账户。全窗财富和配对CI对5个required比较，年度/rolling门相对SOXX。首尾partial年度单列。

FINRA ShortVolume包含ShortExemptVolume，CNMS仅场外RTH子集；q不是shortinterest、情绪、净库存或作者operationalshorting复现。18:00仅历史可用性假设，当前字节修订与首次接收未认证。结果不能授予PIT或前向资格。

采样回撤不认证全分钟风险，重复市场历史不独立。生产未改，稳定超额目标未完成。

关闭固定q_v1，不反号、平滑、择窗口、调损失或控制器救援。

Qlib Recorder：`8e87af53d57a4022bcf19b5dbc690e33`；来源与NAS身份见summary.json、verification_manifest.json、nas_binding.json。

完整年度及区间差异（2bp）：
2024年C净收益1.171%，SOXX为17.428%；相对财富0.861554。
2025年C净收益14.046%，SOXX为43.691%；相对财富0.793690。
C相对SOXX的126日滚动正超额比例在0/2/5bp下仅29.18%/16.97%/6.11%；15个预定配对置信下界均不大于0。q相对同背景B的全窗财富在三个成本均有改善，但背景B显著落后于原shared B，不能把此弱对照改善认定为稳定超额。
5bp下C收盘最大回撤−30.434%、采样点−31.308%，30%风险门失败；CVaR相对SOXX门通过。成本改变独立账户状态路径，收益差并非单纯费用相减。
独立结果评价已接受关闭固定版本：ACCEPT_NOT_PROMOTED_FIXED_EXPERIMENT；执行者未自签晋升，整体Goal未完成。

一次固定失败归因（仅本轮Recorder）：2bp下，C在全部4个年度（2完整、2partial）均胜背景B，但全部年度均不及原shared B及SOXX；16季度分别胜10/16、4/16、4/16，完整14季度与partial2季度另列。所有3成本的180项年/季度配对保存在failure_period_comparison.csv。

C平滑提议平均名义beta为0.866，低于背景B的0.905；2bp实际持仓beta反而为0.597，高于B的0.506，风险非零日641对761。同成本B/C风险状态227日不同。原shared B实际beta为0.849、现金46.43%，新背景B为0.506/62.94%、C为0.597/58.71%。这些是不同决策/持仓时点及内生路径的描述，不能相加为精确cash drag分解，也不能归因于单一controller。

C在0对2、0对5、2对5bp之间分别有200、524、552日风险状态不同，费用与自身状态反馈共同影响结果。6个新账户各3778个点的原生核对最大相对误差均为0；没有发现可以解释失败的账务实现误差。原shared B的raw/smoothed及逐日riskstate未复制进本轮Recorder，本次未额外读旧源或倒造反事实。详见failure_attribution.json和小表。
独立交付读回：NAS冷恢复102个产物的大小及哈希全部一致，原生summary/gates、主/辅助/来源Dataset直接prepare及6个已存原生账本通过。0重拟合、0账户重放。详见independent_evaluation.json与readback_verification.json。
